In [ ]:
# Instalar dependencias
#!pip install -q wikipedia-api sentence-transformers chromadb langchain langchain-community
#!pip install -q transformers torch pandas numpy markdown

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# Importaciones
import wikipediaapi
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import json
import os
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas correctamente")

c:\Users\atara\Desktop\rag_wikipedia-lab\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Librerías importadas correctamente


In [2]:

# Crear estructura de directorios
os.makedirs('data', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
os.makedirs('notebooks', exist_ok=True)
print("✅ Directorios creados: data/, outputs/, notebooks/")

✅ Directorios creados: data/, outputs/, notebooks/


**1️⃣ CREACIÓN DEL CONJUNTO DE DATOS DE WIKIPEDIA**

In [3]:

print("\n" + "="*60)
print("1️⃣ EXTRACCIÓN DE DATOS DE WIKIPEDIA")
print("="*60 + "\n")

# Configurar Wikipedia API
wiki = wikipediaapi.Wikipedia(
    user_agent='RAGProject/1.0 (Educational Purpose)',
    language='en'
)

# Obtener página sobre Federated Learning
page = wiki.page("Federated_learning")

if not page.exists():
    print("❌ La página no existe, intentando con título alternativo...")
    page = wiki.page("Federated_machine_learning")

if page.exists():
    print(f"✅ Página encontrada: {page.title}")
    print(f"📏 Longitud del texto: {len(page.text)} caracteres")
    print(f"🔗 URL: {page.fullurl}")
    full_text = page.text
else:
    print("⚠️  No se pudo cargar de Wikipedia, usando texto de ejemplo...")


1️⃣ EXTRACCIÓN DE DATOS DE WIKIPEDIA

✅ Página encontrada: Federated learning
📏 Longitud del texto: 31699 caracteres
🔗 URL: https://en.wikipedia.org/wiki/Federated_learning


In [4]:
# Función para segmentar texto en chunks con overlap
def chunk_text(text: str, words_per_chunk: int = 300, overlap: int = 50) -> List[str]:
    """
    Divide el texto en chunks con overlap para mantener contexto.

    Args:
        text: Texto a segmentar
        words_per_chunk: Número de palabras por chunk (objetivo: ~300)
        overlap: Número de palabras de overlap entre chunks

    Returns:
        Lista de chunks de texto
    """
    # Limpiar el texto
    text = ' '.join(text.split())  # Normalizar espacios
    words = text.split()
    chunks = []

    i = 0
    while i < len(words):
        # Tomar palabras para este chunk
        chunk_words = words[i:i + words_per_chunk]
        chunk = ' '.join(chunk_words)

        # Solo agregar chunks con contenido significativo
        if len(chunk_words) > 50:  # Filtrar chunks muy pequeños
            chunks.append(chunk)

        # Avanzar con overlap
        i += (words_per_chunk - overlap)

    return chunks

# Extraer y segmentar el texto
print("\n⏳ Segmentando texto en chunks de ~300 palabras...")
chunks = chunk_text(full_text, words_per_chunk=300, overlap=50)

print(f"✅ Texto segmentado en {len(chunks)} chunks")
word_counts = [len(c.split()) for c in chunks]
print(f"📊 Promedio de palabras por chunk: {np.mean(word_counts):.1f}")
print(f"📊 Rango: {min(word_counts)} - {max(word_counts)} palabras")


⏳ Segmentando texto en chunks de ~300 palabras...
✅ Texto segmentado en 17 chunks
📊 Promedio de palabras por chunk: 296.5
📊 Rango: 240 - 300 palabras


In [5]:
# Crear DataFrame con metadatos
print("\n⏳ Creando corpus con metadatos...")
data = []
for i, chunk in enumerate(chunks):
    data.append({
        'id': f'chunk_{i:03d}',
        'title': f'Federated Learning - Section {i+1}',
        'text': chunk,
        'word_count': len(chunk.split()),
        'source': 'Wikipedia',
        'page_title': 'Federated Learning'
    })

df = pd.DataFrame(data)

# Guardar corpus
corpus_path = 'data/wiki_corpus.csv'
df.to_csv(corpus_path, index=False)
print(f"✅ Corpus guardado en {corpus_path}")
print(f"\n📋 Estructura del corpus:")
print(df[['id', 'title', 'word_count']].head(5))


⏳ Creando corpus con metadatos...
✅ Corpus guardado en data/wiki_corpus.csv

📋 Estructura del corpus:
          id                           title  word_count
0  chunk_000  Federated Learning - Section 1         300
1  chunk_001  Federated Learning - Section 2         300
2  chunk_002  Federated Learning - Section 3         300
3  chunk_003  Federated Learning - Section 4         300
4  chunk_004  Federated Learning - Section 5         300


**INSERCIÓN + ALMACENAMIENTO VECTORIAL**

In [6]:
print("\n" + "="*60)
print("2️⃣ EMBEDDINGS Y ALMACENAMIENTO VECTORIAL")
print("="*60 + "\n")

# Cargar modelo de embeddings
print("⏳ Cargando modelo SentenceTransformer...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✅ Modelo cargado: all-MiniLM-L6-v2")
print(f"📐 Dimensiones de embedding: {embedding_model.get_sentence_embedding_dimension()}")

# Inicializar ChromaDB con persistencia
print("\n⏳ Inicializando ChromaDB...")
try:
    # Nuevo cliente persistente
    chroma_client = chromadb.PersistentClient(path="./chroma_db")
    print("✅ ChromaDB inicializado con persistencia en './chroma_db'")
except Exception as e:
    print("❌ Error inicializando PersistentClient, usando fallback temporal:", e)
    chroma_client = chromadb.Client()  # fallback simple

collection_name = "wiki_federated_learning"
# Eliminar colección si existe (para re-ejecutar)
try:
    chroma_client.delete_collection(collection_name)
    print("🗑️  Colección anterior eliminada")
except:
    pass

# Crear colección nueva
collection = chroma_client.create_collection(
    name=collection_name,
    metadata={
        "description": "Wikipedia Federated Learning chunks",
        "model": "all-MiniLM-L6-v2",
        "chunk_size": "~300 words"
    }
)

print(f"✅ Colección '{collection_name}' creada\n")


2️⃣ EMBEDDINGS Y ALMACENAMIENTO VECTORIAL

⏳ Cargando modelo SentenceTransformer...
✅ Modelo cargado: all-MiniLM-L6-v2
📐 Dimensiones de embedding: 384

⏳ Inicializando ChromaDB...
✅ ChromaDB inicializado con persistencia en './chroma_db'
✅ Colección 'wiki_federated_learning' creada



In [7]:
# Generar embeddings e insertar en ChromaDB
print("⏳ Generando embeddings e insertando en ChromaDB...\n")

batch_size = 10
total_batches = (len(df) + batch_size - 1) // batch_size

for batch_num in range(total_batches):
    start_idx = batch_num * batch_size
    end_idx = min(start_idx + batch_size, len(df))
    batch = df.iloc[start_idx:end_idx]

    # Generar embeddings
    embeddings = embedding_model.encode(
        batch['text'].tolist(),
        show_progress_bar=False
    ).tolist()

    # Preparar metadatos
    metadatas = [
        {
            'title': row['title'],
            'word_count': int(row['word_count']),
            'source': row['source'],
            'page_title': row['page_title']
        }
        for _, row in batch.iterrows()
    ]

    # Insertar en ChromaDB
    collection.add(
        embeddings=embeddings,
        documents=batch['text'].tolist(),
        metadatas=metadatas,
        ids=batch['id'].tolist()
    )

    print(f"  ✓ Batch {batch_num + 1}/{total_batches}: Chunks {start_idx+1}-{end_idx} insertados")

print(f"\n✅ Total de {len(df)} chunks insertados en ChromaDB")
print(f"📊 Verificación - Colección contiene: {collection.count()} documentos")


⏳ Generando embeddings e insertando en ChromaDB...

  ✓ Batch 1/2: Chunks 1-10 insertados
  ✓ Batch 2/2: Chunks 11-17 insertados

✅ Total de 17 chunks insertados en ChromaDB
📊 Verificación - Colección contiene: 17 documentos


**PIPELINE DE CONSULTAS RAG**

In [8]:
print("\n" + "="*60)
print("3️⃣ PIPELINE DE RECUPERACIÓN Y GENERACIÓN")
print("="*60 + "\n")

# Función para realizar búsqueda vectorial
def search_similar_chunks(query: str, top_k: int = 5) -> Dict:
    """
    Busca los chunks más similares a la consulta usando búsqueda vectorial.

    Args:
        query: Pregunta del usuario
        top_k: Número de resultados a retornar

    Returns:
        Diccionario con resultados de la búsqueda
    """
    # Generar embedding de la consulta
    query_embedding = embedding_model.encode(query).tolist()

    # Buscar en ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return {
        'query': query,
        'documents': results['documents'][0],
        'metadatas': results['metadatas'][0],
        'distances': results['distances'][0],
        'ids': results['ids'][0]
    }

# Función para generar resumen RAG
def generate_rag_summary(query: str, retrieved_chunks: List[str], use_ollama: bool = False) -> str:
    """
    Genera un resumen usando los chunks recuperados.

    Args:
        query: Pregunta del usuario
        retrieved_chunks: Lista de chunks recuperados
        use_ollama: Si True, intenta usar Ollama (requiere instalación local)

    Returns:
        Resumen generado
    """
    # Combinar chunks recuperados
    context = "\n\n".join([f"Context {i+1}:\n{chunk}" for i, chunk in enumerate(retrieved_chunks)])

    if use_ollama:
        try:
            from langchain_community.llms import Ollama
            from langchain.chains import RetrievalQA
            from langchain.prompts import PromptTemplate

            # Configurar Ollama
            llm = Ollama(model="mistral")

            # Crear prompt
            prompt_template = """Use the following pieces of context to answer the question at the end.
            If you don't know the answer, just say that you don't know, don't try to make up an answer.
            Provide a comprehensive answer in 400-500 words.

            Context:
            {context}

            Question: {question}

            Detailed Answer:"""

            # Generar respuesta
            full_prompt = prompt_template.format(context=context, question=query)
            response = llm(full_prompt)
            return response

        except Exception as e:
            print(f"⚠️  Error usando Ollama: {e}")
            print("📝 Generando resumen simulado...\n")
            # Fallback if Ollama fails
            return f"[Simulated RAG Summary - Ollama failed or not available]\nQuery: {query}\nContexts used: {len(retrieved_chunks)}\nFirst 200 chars of context: {context[:200]}..."
    else:
        # Fallback when use_ollama is False
        return f"[Simulated RAG Summary - Ollama not used]\nQuery: {query}\nContexts used: {len(retrieved_chunks)}\nFirst 200 chars of context: {context[:200]}..."



3️⃣ PIPELINE DE RECUPERACIÓN Y GENERACIÓN



In [9]:
# Configuración de consultas de ejemplo
sample_queries = [
    "Explain federated learning challenges in healthcare",
    "What are the privacy benefits of federated learning?",
    "How does federated learning handle data heterogeneity?",
    "What is Federated Averaging algorithm?"
]

print("🔍 Consultas de ejemplo disponibles:")
for i, q in enumerate(sample_queries, 1):
    print(f"  {i}. {q}")

🔍 Consultas de ejemplo disponibles:
  1. Explain federated learning challenges in healthcare
  2. What are the privacy benefits of federated learning?
  3. How does federated learning handle data heterogeneity?
  4. What is Federated Averaging algorithm?


 **EJECUTAR CONSULTAS Y GENERAR RESULTADOS**

In [10]:
print("\n" + "="*60)
print("4️⃣ GENERACIÓN DE RESÚMENES Y EJEMPLOS")
print("="*60 + "\n")

# Seleccionar consulta principal
main_query = sample_queries[0]
print(f"📝 Procesando consulta principal: '{main_query}'\n")

# Realizar búsqueda
print("🔎 Buscando chunks relevantes...")
search_results = search_similar_chunks(main_query, top_k=5)

print(f"✅ Recuperados {len(search_results['documents'])} chunks relevantes\n")

# Mostrar resultados de recuperación
print("📊 Top 3 chunks recuperados:")
for i in range(min(3, len(search_results['documents']))):
    print(f"\n--- Chunk {i+1} (ID: {search_results['ids'][i]}) ---")
    print(f"Título: {search_results['metadatas'][i]['title']}")
    print(f"Distancia: {search_results['distances'][i]:.4f}")
    print(f"Texto: {search_results['documents'][i][:200]}...")

# Generar resumen RAG
print("\n\n⏳ Generando resumen RAG...")
rag_summary = generate_rag_summary(
    main_query,
    search_results['documents'][:3],
    use_ollama=False  # Cambiar a True si tienes Ollama instalado
)

print("✅ Resumen generado\n")
print("="*60)
print(rag_summary)
print("="*60)

# Guardar resumen principal
summary_path = 'outputs/rag_summary.md'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(f"# RAG Summary: Federated Learning\n\n")
    f.write(f"## Query: {main_query}\n\n")
    f.write(f"### Summary\n\n")
    f.write(rag_summary)
    f.write(f"\n\n### Retrieved Context\n\n")
    for i, doc in enumerate(search_results['documents'][:3], 1):
        f.write(f"#### Context {i}\n")
        f.write(f"**Source:** {search_results['metadatas'][i-1]['title']}\n\n")
        f.write(f"{doc}\n\n")
    f.write(f"\n---\n")
    f.write(f"*Generated using RAG pipeline with ChromaDB + SentenceTransformers + LangChain*\n")

print(f"\n✅ Resumen guardado en: {summary_path}")

# Procesar múltiples consultas para ejemplos
print("\n⏳ Procesando consultas adicionales...")
all_examples = []

for query in sample_queries:
    print(f"  • {query}")
    results = search_similar_chunks(query, top_k=3)

    example = {
        'query': query,
        'retrieved_chunks': [
            {
                'id': results['ids'][i],
                'title': results['metadatas'][i]['title'],
                'text': results['documents'][i],
                'distance': float(results['distances'][i]),
                'similarity_score': 1 - float(results['distances'][i])
            }
            for i in range(len(results['documents']))
        ]
    }
    all_examples.append(example)

# Guardar ejemplos de recuperación
examples_path = 'outputs/retrieval_examples.json'
with open(examples_path, 'w', encoding='utf-8') as f:
    json.dump(all_examples, f, indent=2, ensure_ascii=False)

print(f"✅ Ejemplos guardados en: {examples_path}")


4️⃣ GENERACIÓN DE RESÚMENES Y EJEMPLOS

📝 Procesando consulta principal: 'Explain federated learning challenges in healthcare'

🔎 Buscando chunks relevantes...
✅ Recuperados 5 chunks relevantes

📊 Top 3 chunks recuperados:

--- Chunk 1 (ID: chunk_015) ---
Título: Federated Learning - Section 16
Distancia: 0.7133
Texto: which the authors explore how federated learning may provide a solution for the future of digital health, and highlight the challenges and considerations that need to be addressed. Recently, a collabo...

--- Chunk 2 (ID: chunk_000) ---
Título: Federated Learning - Section 1
Distancia: 0.7700
Texto: Federated learning (also known as collaborative learning) is a machine learning technique in a setting where multiple entities (often called clients) collaboratively train a model while keeping their ...

--- Chunk 3 (ID: chunk_007) ---
Título: Federated Learning - Section 8
Distancia: 0.8553
Texto: the constraints of the machine learning application (e.g., available computi